In [1]:
import sys
sys.path.append("../ingestion/python/src")
sys.path.append("../ingestion/python/LangGraph_Agent")

from utils import *
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

from silver_enrichment import *
from graph_silver_enrichment import *
from APIendpoint import PlacesAPI
from database import Database

import os
from dotenv import load_dotenv
load_dotenv(override=True)


llm = LLM()
places_api = PlacesAPI(os.getenv('MAPS_APP_KEY'))
db = Database()

find_mails_test = FindMails(llm=llm)

print("✅ Tout est initialisé, find_mails_test est prêt")

✅ Tout est initialisé, find_mails_test est prêt


In [ ]:
import time
from langchain_ollama import ChatOllama

llm_test = ChatOllama(model="llama3.2:latest", temperature=0)

t0 = time.time()
response = llm_test.invoke("Say hello in one word")
print(f"Appel simple : {time.time()-t0:.1f}s → {response.content}")

In [2]:
import time

print("Test 1 : DDG seul, sans LLM")
t0 = time.time()
from ddgs import DDGS
with DDGS() as ddgs:
    results = ddgs.text("Axity careers contact email", max_results=5)
print(f"DDG seul : {time.time()-t0:.1f}s, {len(results)} résultats")

Test 1 : DDG seul, sans LLM
DDG seul : 1.2s, 5 résultats


In [ ]:
print("Test 2 : Appel LLM seul (génération de query)")
t0 = time.time()
query_result = find_mails_test._generate_query({"company_name": "Axity", "city": "Santiago", "country": "CL"})
print(f"Génération query : {time.time()-t0:.1f}s → {query_result}")

Test 2 : Appel LLM seul (génération de query)


In [ ]:
print("Test 3 : Appel LLM seul (extraction emails) avec un texte bidon")
t0 = time.time()
response = find_mails_test._llm_structured.invoke([
    SystemMessage(content="Extract emails from this text."),
    HumanMessage(content="Contact us at info@test.com")
])
print(f"Extraction emails : {time.time()-t0:.1f}s → {response}")